# AERIS: Air & Environmental Risk Intelligence System
## Exploratory Data Analysis, Statistical Profiling, and Environmental Risk Modeling

**Academic Context:** Semester 5 College Mini Project  
**Team:** Team of Four Students  
**Domain:** Environmental Atmospheric Data Analytics & Public Health Engineering  

---
### Notebook Objectives
1. Load historical ambient air quality data across selected Indian urban airsheds.
2. Standardize column nomenclature, remove duplicates, and validate sensor physical bounds.
3. Perform parametric and non-parametric exploratory data analysis (EDA).
4. Analyze cyclical trends and Indian Meteorological Department (IMD) seasonal shifts.
5. Evaluate inter-pollutant correlation structures (Pearson & Spearman).
6. Compute the AERIS Cumulative Exposure Indicator (CEI) and identify chronic pollution hotspots.

In [ ]:
# 1. Environment Setup and Library Ingestion
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import load_dataset, clean_air_quality_data, NAAQS_STANDARDS
from src.analysis import calculate_descriptive_statistics, compute_correlation_matrix
from src.risk_analysis import calculate_cumulative_exposure_indicator, identify_pollution_hotspots, detect_unusual_observations

# Set plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 120
print('Libraries successfully imported.')

## 2. Data Ingestion and Quality Validation
We load raw air quality observations and audit data quality using the AERIS preprocessing pipeline.

In [ ]:
raw_csv_path = '../data/raw/india_city_air_quality.csv'
df_raw = load_dataset(raw_csv_path)
print(f'Raw observations loaded: {len(df_raw):,}')

df_clean, summary = clean_air_quality_data(df_raw)
print('Data Cleaning Audit:')
for k, v in summary.to_dict().items():
    print(f'  - {k}: {v}')

df_clean.head()

## 3. Parametric & Non-Parametric Descriptive Statistics
We calculate central tendencies (mean, median), dispersion (std dev, IQR), and distribution shape (skewness, kurtosis).

In [ ]:
stats_df = calculate_descriptive_statistics(df_clean, pollutants=['PM2.5', 'PM10', 'NO2', 'SO2'], group_by='City')
stats_df.head(10)

## 4. Seasonal & Dispersion Analysis (Seaborn & Matplotlib)
Evaluating the impact of IMD meteorological seasons (Winter, Summer, Monsoon, Post-Monsoon) on PM2.5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Seasonal Boxplot
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
sns.boxplot(
    data=df_clean, x='Season', y='PM2.5', order=season_order,
    palette=['#3B82F6', '#F59E0B', '#10B981', '#EF4444'], ax=axes[0]
)
axes[0].axhline(60, color='red', linestyle='--', linewidth=1.5, label='NAAQS Limit (60 µg/m³)')
axes[0].set_title('PM2.5 Dispersion across IMD Seasons', fontsize=12, fontweight='bold')
axes[0].set_ylabel('PM2.5 Concentration (µg/m³)')
axes[0].legend()

# City-wise Distribution
sns.barplot(
    data=df_clean, x='City', y='PM2.5', estimator=np.mean, errorbar='sd',
    palette='crest', ax=axes[1]
)
axes[1].axhline(60, color='red', linestyle='--', linewidth=1.5, label='NAAQS Limit (60 µg/m³)')
axes[1].set_title('City-wise Mean PM2.5 (with Std Dev)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean PM2.5 (µg/m³)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Multi-Pollutant Correlation Matrix
Examining collinearity between particulate matter (PM2.5, PM10) and gaseous precursors (NO2, SO2, CO, O3).

In [ ]:
corr_matrix, p_matrix = compute_correlation_matrix(df_clean)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1.0, vmax=1.0, linewidths=0.5, cbar_kws={'label': 'Pearson Correlation (r)'}
)
plt.title('Inter-Pollutant Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Environmental Risk Intelligence: Hotspots & Cumulative Exposure Indicator (CEI)
Computing the weighted cumulative multi-pollutant exposure score normalized against CPCB standards.

In [ ]:
# Cumulative Exposure Indicator (CEI)
cei_df, metadata = calculate_cumulative_exposure_indicator(df_clean, group_by='City')
print('AERIS CEI Standings (100 = NAAQS Exact Compliance):')
print(cei_df[['Exposure Rank', 'City', 'AERIS CEI Score', 'Risk Tier', 'Data Completeness (%)']])

# Hotspot Identification
hotspots = identify_pollution_hotspots(df_clean, primary_pollutant='PM2.5')
print('\nUrban Hotspots Leaderboard:')
print(hotspots[['Rank', 'City', 'Hotspot Score', 'Hotspot Status', 'Exceedance Rate (%)']])

# Statistical Anomaly Detection
anomalies = detect_unusual_observations(df_clean, pollutant='PM2.5', method='iqr')
print(f'\nDetected {len(anomalies)} statistical anomaly events (severe episodic peaks).')
anomalies.head(5)

## 7. Conclusions & Findings
- **Indo-Gangetic Airsheds:** Cities such as Delhi and Lucknow demonstrate critical hotspot categorization with CEI scores exceeding 170.
- **Seasonal Disparity:** Winter particulate concentrations exceed monsoon levels by >2x due to temperature inversions and wind stagnation.
- **High Particulate Correlation:** Fine and coarse particulates exhibit strong collinearity ($r > 0.85$), indicative of shared combustion and road dust resuspension.
- **Methodological Note:** CEI serves as a comparative atmospheric risk tool, distinct from clinical medical diagnostics.